# Import packages and load data

In [ ]:
import pandas as pd
import numpy as np
import time
import sklearn
from sklearn.ensemble import RandomForestRegressor
from skopt import BayesSearchCV
from skopt.space import Integer, Real, Categorical
from sklearn.model_selection import train_test_split
from sklearn import model_selection
from sklearn.preprocessing import StandardScaler
from joblib import dump, load

# Data pre-processing

In [ ]:
# Create a loop that stores each data set into a dictionary
output = 'gisaxs_domain'
data_directory = '../../chemical-featurization/likeness_data/poly_morgan.xlsx'
df_likeness = pd.read_excel(data_directory)

# Bayesian-Search an uncertainty sampling

In [ ]:
def bayes_train(df, output):

    df_clean = df.dropna()
    inputs = df_clean.iloc[:, 1:-2].copy()
    target = df_clean[f"{output}"]

    X_train, X_test, y_train, y_test = train_test_split(
        inputs, target, test_size=0.2, random_state=10
    )

    RANDOM_SEED = 42
    CV_FOLDS = 5

    search_spaces = {

        # forest size (log scaling explores wide range efficiently)
        "n_estimators": Integer(50, 600),

        # tree complexity
        "max_depth": Integer(3, 60),

        # regularization knobs
        "min_samples_split": Integer(2, 50),
        "min_samples_leaf": Integer(1, 25),

        # feature sampling
        "max_features": Categorical(["sqrt", "log2", 1.0, 0.3, 0.5, 0.8]),

        # node control
        "max_leaf_nodes": Integer(10, 200),

        # bootstrap behavior
        "bootstrap": Categorical([True, False]),

        # split criterion
        "criterion": Categorical(["squared_error", "friedman_mse"]),
    }

    rf = RandomForestRegressor(
        random_state=RANDOM_SEED,
        n_jobs=-1
    )

    opt = BayesSearchCV(
        estimator=rf,
        search_spaces=search_spaces,
        n_iter=60,                 # number of Bayesian steps
        scoring="r2",
        cv=CV_FOLDS,
        n_jobs=-1,
        random_state=RANDOM_SEED,
        verbose=0
    )

    # Fit optimizer
    opt.fit(X_train, y_train)

    best_model = opt.best_estimator_
    best_cv_r2 = opt.best_score_
    best_test_r2 = best_model.score(X_test, y_test)

    print("CV Train R²:", best_cv_r2)
    print("Test R²:", best_test_r2)
    print("Best Parameters:", opt.best_params_)

    return best_model

In [ ]:
def pseudo_active_learning_rfr(df, output, n_iterations=250):

    df_clean = df.dropna()
    inputs = df_clean.iloc[:, 1:-2].copy()
    target = df_clean[f"{output}"]
    X_train_full, X_test, y_train_full, y_test = train_test_split(
        inputs, target, test_size=0.2, random_state=10
    )

    best_model = bayes_train(df, output)
    best_params = best_model.get_params()
    best_params.pop("random_state", None)

    initial_seeds = [1221]
    total_seeds = 10
    rng_seed_gen = np.random.default_rng(42)
    remaining = total_seeds - len(initial_seeds)
    if remaining < 0:
        raise ValueError("You provided more seeds than the total requested.")
    random_seeds = set()
    while len(random_seeds) < remaining:
        candidate = int(rng_seed_gen.integers(0, 10_000))
        if candidate not in initial_seeds:
            random_seeds.add(candidate)
    all_seeds = initial_seeds + list(random_seeds)
    assert len(all_seeds) == total_seeds

    n_initial = 10
    n_samples = len(X_train_full)

    ensemble_seeds = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

    all_avg_test_r2_history = {}
    all_std_test_r2_history = {}

    for seed in all_seeds:

        np.random.seed(seed)
        initial_indices = np.random.choice(n_samples, n_initial, replace=False)

        train_idx = list(initial_indices)
        pool_idx = list(set(np.arange(n_samples)) - set(train_idx))

        avg_test_r2_history = []
        std_test_r2_history = []

        for iteration in range(n_iterations):
            X_train = X_train_full.iloc[train_idx]
            y_train = y_train_full.iloc[train_idx]
            X_pool = X_train_full.iloc[pool_idx]

            models = []
            test_r2_scores = []
            pool_predictions = []

            for ensemble_seed in ensemble_seeds:
                model = RandomForestRegressor(
                    **best_params,
                    random_state=ensemble_seed
                )
                model.fit(X_train, y_train)
                models.append(model)

                y_pred_test = model.predict(X_test)
                test_r2_scores.append(r2_score(y_test, y_pred_test))
                pool_predictions.append(model.predict(X_pool))

            pool_predictions = np.vstack(pool_predictions)
            uncertainty = np.std(pool_predictions, axis=0)

            selected_pool_position = np.argmax(uncertainty)
            selected_index = pool_idx[selected_pool_position]

            train_idx.append(selected_index)
            pool_idx.remove(selected_index)

            avg_test_r2 = np.mean(test_r2_scores)
            std_test_r2 = np.std(test_r2_scores)
            avg_test_r2_history.append(avg_test_r2)
            std_test_r2_history.append(std_test_r2)

            if iteration % 25 == 0:
                print(f"Seed {seed} | Iteration {iteration:3d} | Avg Test R$^2$: {avg_test_r2:.4f}")

        all_avg_test_r2_history[seed] = avg_test_r2_history
        all_std_test_r2_history[seed] = std_test_r2_history

    results = {}
    for seed in all_seeds:
        results[seed] = pd.DataFrame({
            "iteration": np.arange(len(all_avg_test_r2_history[seed])),
            "avg_test_r2": all_avg_test_r2_history[seed],
            "std_test_r2": all_std_test_r2_history[seed]
        })

    return results

In [ ]:
%%time

results = pseudo_active_learning_rfr(df_likeness, output, 250)

In [ ]:
# save to Excel
results.to_excel("rfr_domain_uncertainty.xlsx", index=False)